In [1]:
%pwd

'c:\\Customer_churn_project\\notebooks'

In [2]:
import os

os.chdir("../")

In [3]:
from dataclasses import dataclass
from pathlib import Path

In [4]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    input_data_path: Path
    base_model_path: Path
    tuned_model_path: Path
    scores_file_path: Path
    base_cm_path: Path
    tuned_cm_path: Path
    roc_curve_path:Path

In [5]:
from src.CustomerChurnPrediction.constants import *
from src.CustomerChurnPrediction.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH):
        self.config = read_yaml(config_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self):

        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            input_data_path=config.input_data_path,
            base_model_path=config.base_model_path,
            tuned_model_path=config.tuned_model_path,
            scores_file_path=config.scores_file_path,
            base_cm_path = config.base_cm_path,
            tuned_cm_path=config.tuned_cm_path,
            roc_curve_path=config.roc_curve_path
        )

        return model_evaluation_config

In [7]:
import dagshub
dagshub.init(repo_owner='udaypatel2209', repo_name='Customer-Churn-Prediction', mlflow=True)

[2026-06-22 01:54:25,401] 1025 httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"


Accessing as udaypatel2209

[2026-06-22 01:54:25,422] 107 dagshub - INFO - Accessing as udaypatel2209
[2026-06-22 01:54:33,287] 1025 httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/udaypatel2209/Customer-Churn-Prediction "HTTP/1.1 200 OK"
[2026-06-22 01:54:39,330] 1025 httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"


Initialized MLflow to track repo "udaypatel2209/Customer-Churn-Prediction"

[2026-06-22 01:54:39,337] 107 dagshub - INFO - Initialized MLflow to track repo "udaypatel2209/Customer-Churn-Prediction"


Repository udaypatel2209/Customer-Churn-Prediction initialized!

[2026-06-22 01:54:39,343] 107 dagshub - INFO - Repository udaypatel2209/Customer-Churn-Prediction initialized!


In [8]:
import mlflow

mlflow.set_registry_uri("https://dagshub.com/udaypatel2209/Customer-Churn-Prediction.mlflow")
mlflow.set_tracking_uri("https://dagshub.com/udaypatel2209/Customer-Churn-Prediction.mlflow")
mlflow.set_experiment("Telco Churn - Baseline Models")

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/1fbc64183f474e7f971dda6feae71bb0', creation_time=1781947070965, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1781947070965, lifecycle_stage='active', name='Telco Churn - Baseline Models', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [9]:
import time
import json
import pandas as pd
import optuna
import joblib
import seaborn as sns
import mlflow
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)


In [10]:
import os
import sys
from dotenv import load_dotenv
from sqlalchemy import create_engine
from src.CustomerChurnPrediction.utils.logger import logger
from src.CustomerChurnPrediction.utils.exception import CustomException

load_dotenv()

True

In [11]:
class ModelEvaluation:
    """Evaluates the base and tuned models on a held-out test set, logs results to MLflow, and saves scores to JSON."""

    THRESHOLD = 0.25

    def __init__(self, config: ModelEvaluationConfig):
        """Sets up config and connects MLflow tracking to DagsHub."""
        self.config = config

        dagshub.init(
            repo_owner=os.getenv("DAGSHUB_REPO_OWNER"),
            repo_name=os.getenv("DAGSHUB_REPO_NAME"),
            mlflow=True,
        )

    def get_input_data(self) -> pd.DataFrame:
        """Loads the transformed data used for training."""
        logger.info(f"Loading data from: {self.config.input_data_path}")
        return pd.read_csv(self.config.input_data_path)

    def split_data(self, df: pd.DataFrame, target_col: str = "Churn", test_size: float = 0.2):
        """Reproduces the same train/test split used during training (same seed)."""
        X = df.drop(columns=[target_col])
        y = df[target_col]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        return X_train, X_test, y_train, y_test

    def load_models(self):
        """Loads the saved base and tuned models from disk."""
        base_model = joblib.load(self.config.base_model_path)
        tuned_model = joblib.load(self.config.tuned_model_path)
        logger.info("Loaded base and tuned models")
        return base_model, tuned_model

    def evaluate_model(self, model, X_test, y_test):
        """Computes precision, recall, f1, and roc_auc for a model at the fixed threshold."""
        proba = model.predict_proba(X_test)[:, 1]
        y_pred = (proba >= self.THRESHOLD).astype(int)

        metrics = {
            "precision": precision_score(y_test, y_pred, pos_label=1),
            "recall": recall_score(y_test, y_pred, pos_label=1),
            "f1": f1_score(y_test, y_pred, pos_label=1),
            "roc_auc": roc_auc_score(y_test, proba),
        }

        logger.info(f"Metrics: {metrics}")
        logger.info("\n" + classification_report(y_test, y_pred, digits=3))

        return metrics, y_pred

    def log_confusion_matrix(self, title: str, save_path: str, y_test, y_pred) -> str:
        """Builds a confusion matrix plot for a model and saves it as a PNG."""
        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title(f"Confusion Matrix - {title}")

        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path)
        plt.close()

        logger.info(f"Confusion matrix saved to {save_path}")
        return save_path
    
    def plot_roc_curve(self, base_model, tuned_model, X_test, y_test) -> None:
        """Saves an ROC curve plot comparing the base and tuned models."""
        fig, ax = plt.subplots(figsize=(7, 6))

        for name, model in [("Base XGBoost", base_model), ("Tuned XGBoost", tuned_model)]:
            proba = model.predict_proba(X_test)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, proba)
            auc = roc_auc_score(y_test, proba)
            ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

        ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC Curve -- Base vs Tuned XGBoost")
        ax.legend(loc="lower right")
        ax.grid(True)

        Path(self.config.roc_curve_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(self.config.roc_curve_path, bbox_inches="tight")
        plt.close(fig)
        logger.info(f"ROC curve saved to {self.config.roc_curve_path}")

    def log_to_mlflow(self, run_name: str, model, metrics: dict, cm_path: str,roc_path: str) -> None:
        """Logs a model's hyperparameters, evaluation metrics, and confusion matrix to MLflow."""
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params(model.get_params())
            mlflow.log_param("threshold", self.THRESHOLD)
            mlflow.log_metrics(metrics)
            mlflow.log_artifact(cm_path)
            mlflow.log_artifact(roc_path)

        logger.info(f"Logged '{run_name}' evaluation run to MLflow")

    def save_scores(self, scores: dict) -> None:
        """Saves all models' metrics to a single JSON file."""
        os.makedirs(os.path.dirname(self.config.scores_file_path), exist_ok=True)

        with open(self.config.scores_file_path, "w") as f:
            json.dump(scores, f, indent=4)

        logger.info(f"Scores saved to {self.config.scores_file_path}")

    def run_evaluation(self) -> None:
        """Runs the full evaluation pipeline: load data/models, evaluate both, log to MLflow, save scores.json."""
        df = self.get_input_data()
        _, X_test, _, y_test = self.split_data(df)

        base_model, tuned_model = self.load_models()

        base_metrics, base_pred = self.evaluate_model(base_model, X_test, y_test)
        tuned_metrics, tuned_pred = self.evaluate_model(tuned_model, X_test, y_test)

        base_cm_path = self.log_confusion_matrix(
            "XGBoost", self.config.base_cm_path, y_test, base_pred
        )
        tuned_cm_path = self.log_confusion_matrix(
            "XGBoost_Tuned", self.config.tuned_cm_path, y_test, tuned_pred
        )
        self.plot_roc_curve(base_model, tuned_model, X_test, y_test)
        # self.log_to_mlflow("XGBoost_Eval", base_model, base_metrics, base_cm_path)
        # self.log_to_mlflow("XGBoost_Tuned_Eval", tuned_model, tuned_metrics, tuned_cm_path)

        scores = {
            "base_model": base_metrics,
            "tuned_model": tuned_metrics,
        }
        self.save_scores(scores)


In [12]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(model_evaluation_config)
    model_evaluation.run_evaluation()
except Exception as e:
    raise CustomException(e,sys)

[2026-06-22 01:54:45,354] 33 CustomerChurnPrediction - INFO - yaml file: config\config.yml loaded successfully
[2026-06-22 01:54:45,357] 50 CustomerChurnPrediction - INFO - created directory at: artifacts
[2026-06-22 01:54:45,359] 50 CustomerChurnPrediction - INFO - created directory at: artifacts/model_evaluation
[2026-06-22 01:54:53,052] 1025 httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/udaypatel2209/Customer-Churn-Prediction "HTTP/1.1 200 OK"


Initialized MLflow to track repo "udaypatel2209/Customer-Churn-Prediction"

[2026-06-22 01:54:53,067] 107 dagshub - INFO - Initialized MLflow to track repo "udaypatel2209/Customer-Churn-Prediction"


Repository udaypatel2209/Customer-Churn-Prediction initialized!

[2026-06-22 01:54:53,071] 107 dagshub - INFO - Repository udaypatel2209/Customer-Churn-Prediction initialized!
[2026-06-22 01:54:53,073] 18 CustomerChurnPrediction - INFO - Loading data from: artifacts/data_transformation/transformed_data.csv
[2026-06-22 01:54:53,267] 35 CustomerChurnPrediction - INFO - Loaded base and tuned models
[2026-06-22 01:54:53,293] 50 CustomerChurnPrediction - INFO - Metrics: {'precision': 0.46038863976083705, 'recall': 0.8235294117647058, 'f1': 0.5906040268456376, 'roc_auc': 0.812805752416253}
[2026-06-22 01:54:53,309] 51 CustomerChurnPrediction - INFO - 
              precision    recall  f1-score   support

           0      0.911     0.651     0.759      1033
           1      0.460     0.824     0.591       374

    accuracy                          0.697      1407
   macro avg      0.685     0.737     0.675      1407
weighted avg      0.791     0.697     0.714      1407

[2026-06-22 01:54:53,339] 50 CustomerChurnPrediction - INFO - Metrics: {'precision':